# Neural Networks - Part II

Author: Ethan Robert A. Casin

## Introduction

Consider a 3-layer neural network with $n$ input nodes, $h$ hidden nodes, and $p$ output nodes as described below.

<figure style="text-align:center">
    <img src="figures/multi-output.png" width="70%">
</figure>

What we've discussed so far is a neural network that has a single output. We can of course extend this to have arbitrarily large number of outputs. In this notebook, we'll extend what we've learned from Part 1 to generate multiple outputs. We'll also get acquainted on a neural network as a regressor.

## Training procedure
Let's apply what we've learned from Part 1 and extend it here.



### Forward Propagation

The mappping from the features $\{x_1, x_2, \dots, x_n\}$ to the outputs $\{\Psi_{NN}^1, \Psi_{NN}^2, \dots, \Psi_{NN}^p\}$ can described by the following steps. We can assume that we have a bias term in the hidden and output layers. We can denote them as

**STEP 1: Input to the $j$-th hidden node.** 

\begin{equation*}
\Phi_j = h\left(\sum_{i=1}^n f(x_i)w_{ij}  + b_j\right)
\end{equation*}

where $h(\cdot)$ is the hidden node activation function, $\Phi_j$ is the output from the $j$-th hidden node containing a bias term $b$, and $n$ is the total number of input nodes.

**STEP 2: Hidden to the $k$-th output node.**

\begin{equation*}
\Psi_{NN}^k = g\left(\sum_{j=1}^h \Phi_jv_{jk}  + \beta_k\right)
\end{equation*}

where $g(\cdot)$ is the activation function of the output node, $\beta_k$ is the $k$-th bias term, $\Psi_{NN}^k$ is the $k$-th output, where $k\in \{1, 2, \dots, p\}$ and $p$ is the total number of output nodes.

---

**Pause:** In practice, the bias terms $b_j$ and $\beta_k$ are learnable parameters in a neural network. Much like the weights, the bias terms are updated using gradient descent. Refer to the `01-gradient-descent.ipynb` notebook for a sample implementation on how the bias term (or y-intercept discussed there) is optimized.

---

### Backpropagation

As before, we'll need to perform gradient descent to update our parameters (weights and biases) in the neural network. We'll assume that our error or loss function is given by

$$E = \frac{1}{2}\sum_{k=1}^p(\Psi_{NN}^k - \Psi_{\text{actual}}^k)^2$$

We also know that updating the weights from the **hidden to output** nodes, with a learning rate $\gamma_2$, is given by

$$v_{jk}^{t + 1} = v_{jk}^t - \gamma_2\frac{\partial E}{\partial v_{jk}^t}.$$

Meanwhile, the weights for the **input to hidden** nodes, with a learning rate $\gamma_1$, is given by

$$w_{ij}^{t + 1} = w_{ij}^t - \gamma_1\frac{\partial E}{\partial w_{ij}^t}.$$

*For simplicity, let's assume that we have a constant bias for now and we'll not update them.*

---

**Question:** If you were to deduce it, provide the formula for updating the bias terms $b_j$ and $\beta_k$.

---

Answer:

$\beta^{t+1}_k = \beta^t_k - \gamma (\phi_{NN} - \phi_{actual}) * g'\left(\sum_{j=1}^h \Phi_jv_{jk}  + \beta_k\right)$

Suppose in the lecture we have:
$$E = \frac{1}{2} \sum_{k=1}^p (\phi_{NN} - \phi_{act})^2$$

We'll then implement the chain rule that would result in the following:

**Hidden to output**

\begin{align*}
\frac{\partial E}{\partial v_{jk}^t} &= \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)\frac{\partial \Psi_{NN}^k}{\partial v_{jk}^t}\\
\frac{\partial E}{\partial v_{jk}^t} &= \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)g'\left( \sum_{j=1}^h \Phi_jv_{jk}  + \beta_k \right)\Phi_j
\end{align*}

For convenience, we can define our gradient (or our change) for our output layer as 

\begin{equation*}
\Delta \Phi_{O} = \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)g'\left( \sum_{j=1}^h \Phi_jv_{jk}  + \beta_k \right)
\end{equation*}

Therefore, we can rewrite $\frac{\partial E}{\partial v_{jk}^t}$ as

\begin{equation*}
\frac{\partial E}{\partial v_{jk}^t} = \Delta\Phi_{O}\Phi_j
\end{equation*}

This makes our weight change rule then be

$$v_{jk}^{t + 1} = v_{jk}^t - \gamma_2\Delta\Phi_{O}\Phi_j$$


**Input to hidden**

We can do the same logic to update our weights for the input to hidden nodes.

\begin{align*}
\frac{\partial E}{\partial w_{ij}^t} &= \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)\frac{\partial \Psi_{NN}^k}{\partial w_{ij}^t}\\
 &= \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)g'\left( \sum_{j=1}^h \Phi_jv_{jk}  + \beta_k \right)\left(v_{jk}\frac{\partial \Phi_j}{\partial w_{ij}^t} \right)\\
\frac{\partial E}{\partial w_{ij}^t} &= \left( \Psi_{NN}^k - \Psi_{\text{actual}}^k \right)g'\left( \sum_{j=1}^h \Phi_jv_{jk}  + \beta_k \right)\left[v_{jk} h'\left( \sum_{i=1}^n f(x_i)w_{ij}  + b_j \right) \right]f(x_i)\\
\end{align*}

Using the expression $\Delta\Phi_{O}$, we can rewrite the above as

\begin{equation*}
\frac{\partial E}{\partial w_{ij}^t} = \Delta\Phi_{O}\left[v_{jk} h'\left( \sum_{i=1}^n f(x_i)w_{ij}  + b_j \right) \right]f(x_i)
\end{equation*}

We can also simplify our notation by assuming that

$$\Delta\Phi_I = \Delta\Phi_{O}\left[v_{jk} h'\left( \sum_{i=1}^n f(x_i)w_{ij}  + b_j \right) \right]$$

Hence, 

\begin{equation*}
\frac{\partial E}{\partial w_{ij}^t} = \Delta\Phi_{I}f(x_i)
\end{equation*}

Therefore, our weight change rule is

$$w_{ij}^{t + 1} = w_{ij}^t - \gamma_1\Delta\Phi_{I}f(x_i).$$


---

In [1]:
import numpy as np
from sklearn.metrics import accuracy_score


## Example 1: Simultaneous predictions on OR, AND, and XOR

Let's revisit our problem from notebook 1 again, but this time we'll predict all possible combinations simultaneously. Assume again, for convenience, that $f(x)=x$.

|$x_1$ | $x_2$ | $\Psi_{\text{AND}}$| $\Psi_{\text{OR}}$| $\Psi_{\text{XOR}}$|
| ---  | ----  |  -----| --- | --- |
| -1   | -1    | 0| 0 | 0|
|-1    | 1     | 0| 1| 1|
|1     | -1    | 0| 1| 1|
|1     | 1     |1 | 1| 0 |

In [2]:
# Define the activation function
def activation_function(x, deriv=False):
    sigmoid =  1 / (1 + np.exp(-x))
    if deriv:
        return sigmoid * (1 - sigmoid)
    return sigmoid

In [5]:
gamma1 = 0.1  # Input to Hidden
gamma2 = 0.01  # Hidden to Output

n_input_nodes = 3  # No of nodes in the input layer, we have three because we will treat the bias as a separate feature
n_hidden_nodes = 3  # No of nodes in the hidden layer
n_output_nodes = 3  # No of nodes in the output layer

For now, let's only include the bias node on the input to hidden layer. Let's disregard the hidden to output bias nodes for now. We have three output for each phi_and, phi_or, and phi_XOR

In [6]:
X = np.array(
    [
        [-1, -1],
        [-1, 1],
        [1, -1],
        [1, 1]
    ]
)
bias = [1, 1, 1, 1] # These are not learnable in this case
X = np.insert(X, 0, bias, axis=1)

y = np.array([[0, 0, 0],
              [0, 1, 1],
              [0, 1, 1],
              [1, 1, 0]])

In [ ]:
np.random.seed(42)

# Initialize random weights
w1 = 2 * np.random.random((n_input_nodes, n_hidden_nodes)) - 1 

# n_hidden_nodes + 1, because we'll be adding a bias
w2 = 2 * np.random.random((n_hidden_nodes, n_output_nodes)) - 1 

n_iters = int(1e4)

print(f"Initial weights:")
print(f"{w1=}")
print(f"{w2=}")
print("=" * 50)

# train the network 
#layer_0 is just the inputs
for i in range(n_iters):
    layer0 = X
    layer1_input = np.dot(layer0, w1)
    layer1 = activation_function(layer1_input)
    
    layer2_input = np.dot(layer1, w2)
    layer2 = activation_function(layer2_input)

    error = layer2 - y
    if i % 1000 == 0:
        print(f"Iter {i} --- Error: {np.mean(np.abs(error)):.2f}")
    
    layer2_delta = error * activation_function(layer2_input, deriv=True)
    layer1_delta = layer2_delta.dot(w2.T)* activation_function(layer1_input, deriv=True)

    w2 -= gamma2 * layer1.T.dot(layer2_delta)
    w1 -= gamma1 * layer0.T.dot(layer1_delta)

y_preds = np.rint(layer2)
print("=" * 50)
print("Raw Output after training:\n")
print(layer2)
print("Learned weights:\n")
print(f"{w1=}")
print(f"{w2=}")
print("-" * 50)
print(f"Thresholded result at 0.5:\n{y_preds}")
print(f"Actual:\n{y}")
print(f"Accuracy: {accuracy_score(y, y_preds) * 100:.2f}%")

Initial weights:
w1=array([[-0.25091976,  0.90142861,  0.46398788],
       [ 0.19731697, -0.68796272, -0.68801096],
       [-0.88383278,  0.73235229,  0.20223002]])
w2=array([[ 0.41614516, -0.95883101,  0.9398197 ],
       [ 0.66488528, -0.57532178, -0.63635007],
       [-0.63319098, -0.39151551,  0.04951286]])
Iter 0 --- Error: 0.54
Iter 1000 --- Error: 0.37
Iter 2000 --- Error: 0.30
Iter 3000 --- Error: 0.25
Iter 4000 --- Error: 0.21
Iter 5000 --- Error: 0.18
Iter 6000 --- Error: 0.16
Iter 7000 --- Error: 0.14
Iter 8000 --- Error: 0.13
Iter 9000 --- Error: 0.12
Raw Output after training:

[[0.01531289 0.10492401 0.17490375]
 [0.05876747 0.90194647 0.8457815 ]
 [0.05877361 0.90193969 0.84577435]
 [0.88174265 0.96994237 0.21649381]]
Learned weights:

w1=array([[-4.43588188,  0.23628585,  4.5180649 ],
       [-4.5490799 ,  2.86397142, -4.93624609],
       [-4.54892282,  2.86401438, -4.93442824]])
w2=array([[-0.22451833, -2.48714871, -4.08840526],
       [ 2.03294963,  3.48170747, -1.301

## Example 2: Train a neural network to learn the derivatives of a function

NN is a universal function approximator. But in reality, there are a lot of functions that does the same thing. 

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
def nonlin2(x, deriv=False):
    sin = np.sin(x)
    if deriv:
        return np.cos(x)
    return sin  # Fixed to return sin

def nonlin(x,deriv=False):
    linear=2*x
    if(deriv==True):
        return 2.0
    return linear

Let the training functions be given by: $f(x) = 0.1x^2$ and $g(x) = \sin(2x)$. The corresponding derivatives are $f'(x) = 0.2x$ and $g'(x) = 2\cos(2x)$

In [ ]:
X = []
y = []

delta_x = 0.1

for start in np.arange(0, 1, delta_x):
    rangex=np.arange(start, start+4.0, 0.1)

    # g(x)
    X.append(np.sin(2.0*rangex))
    y.append(2.0*np.cos(2.0*rangex))

    # f(x)
    X.append(0.1*rangex**2)
    y.append(0.1*2.0*rangex)



X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=1)

In [ ]:
gamma0=.01 #Input to Hidden
gamma1=.001 #Hidden to Output

no_inputs=len(X[0])
no_hidden=10  #No of hidden nodes
no_outputs=len(y[0])

np.random.seed(42)

# randomly initialize our weights with mean 0
w0 = 2*np.random.random((no_inputs,no_hidden)) - 1
w1 = 2*np.random.random((no_hidden,no_outputs)) - 1

#print(w0)
X=X_train
y=y_train


for iter in range(1000000):

    # Feed forward through layers 0, 1, and 2
    layer0 = X
    layer1 = nonlin2(np.dot(layer0,w0))
    layer2 = nonlin(np.dot(layer1,w1))

    # Error Function
    layer2_error = layer2 - y

    if (iter% 50000) == 0:
        print ("Error =", np.mean(np.abs(layer2_error)))

     #Gradients
    layer2_delta = layer2_error*nonlin(np.dot(layer1,w1),deriv=True)
    layer1_delta = layer2_delta.dot(w1.T)*nonlin2(np.dot(layer0,w0),deriv=True)

    w1 -= gamma1*layer1.T.dot(layer2_delta)
    w0 -= gamma0*layer0.T.dot(layer1_delta)

In [ ]:
plt.figure(figsize=(12,8))

print ("Solid lines are the functions, dashed lines the derivatives, open circles the prediction")

index=10
start=.1*index
rangex=np.arange(start, start+4.0, 0.1)
plt.plot(rangex, X[index],'b-')
plt.plot(rangex, y[index],'b--')
plt.plot(rangex, layer2[index], 'bo',mfc='none')

plt.plot(rangex, X[index+1],'r-')
plt.plot(rangex, y[index+1],'r--')
plt.plot(rangex, layer2[index+1], 'ro',mfc='none')

plt.plot(rangex, X[index+2],'g-')
plt.plot(rangex, y[index+2],'g--')
plt.plot(rangex, layer2[index+2], 'go',mfc='none')

plt.plot(rangex, X[index+3],'y-')
plt.plot(rangex, y[index+3],'y--')
plt.plot(rangex, layer2[index+3], 'yo',mfc='none')


plt.xlabel('x-axis')
plt.ylabel('mapping')
plt.axis('tight')
plt.show()

---

**Activity**

What are some experiments you could do to improve the accuracy of the model? Implement and report.

---

### Practice

Derive the forward and backpropagation of the network below. Include the full equations of the update rules.

Assume that $f(x) = \sin(x)$, $h(x) = \sin(x)$, and $g(x) = \cos(x)$. The learning rates are arbitrary, so define $\gamma$ appropriately. Assume that the bias is constant.

<figure align="center">
<img src="figures/with-hidden.png" width="80%">
</figure>

### Practice: Hard Mode

Derive the forward and backpropagation of the network below. Include the full equations of the update rules.

Assume that $f(x) = \sin(x)$, $h(x) = \sin(x)$, and $g(x) = \cos(x)$. The learning rates are arbitrary, so define $\gamma$ appropriately. **The bias is learnable.**

<figure align="center">
<img src="figures/hard-network.png" width="80%">
</figure>

# Forward